In [1]:
from sample import *
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pandas.api.types import CategoricalDtype
from collections import OrderedDict

In [2]:
ds = ['OU-HiC-N6-pfib-D4','OU-HiC-N6IPSC2-R1','OU-HiC-N6IPSC2MND45-R2']

In [3]:
#combine all EVs ( if you want all bins saneky use this)
#EV = pd.DataFrame()
EV = pd.read_csv("GitHub/Sankeyplot/"+ds[0]+"_100kb_arms.cis.vecs.tsv",sep='\t')[['chrom','start','end']]
for i in range(len(ds)):
    df = pd.read_csv("GitHub/Sankeyplot/"+ds[i]+"_100kb_arms.cis.vecs.tsv",sep='\t')
    #df.loc[df['E1'] > 0, 'New'] = 'A'
    #df.loc[df['E1'] <= 0, 'New'] = 'B'
    EV[ds[i]]=df["E1"]
EVlen=len(EV)
print(len(EV))    
#EV=EV.dropna()
print(len(EV))
#EV.head(20)

28760
28760


In [4]:
flips=EV
print(len(flips))
flips['change']=''
for i in range(len(ds)):
    n=ds[i]
    flips['New']='N'
    flips.loc[flips[n] > 0, 'New'] = 'A'
    flips.loc[flips[n] < 0, 'New'] = 'B'
    flips['change'] = flips['change']+flips['New']
    flips[n.replace("OU-HiC-","")]=flips['change']
    #flips[n]=flips['change']
    #print(flips.head())
flips=flips.drop(columns=ds+['New'])  
print(flips.head())
flips = flips[flips.change != "N"*len(ds)]
print(len(flips))
#flips.to_csv("Flips/Flips-Change.csv",sep='\t',index=False)
grp=flips.groupby('change',as_index = False).count()
#print(grp[grp['change'].str.contains("N")]['start'].sum())
grp.tail()

28760
  chrom   start     end change N6-pfib-D4 N6IPSC2-R1 N6IPSC2MND45-R2
0  chr1       0  100000    NNN          N         NN             NNN
1  chr1  100000  200000    NNN          N         NN             NNN
2  chr1  200000  300000    NNN          N         NN             NNN
3  chr1  300000  400000    NNN          N         NN             NNN
4  chr1  400000  500000    NNN          N         NN             NNN
25994


,change,chrom,start,end,N6-pfib-D4,N6IPSC2-R1,N6IPSC2MND45-R2
18,NBA,2,2,2,2,2,2
19,NBB,8,8,8,8,8,8
20,NBN,1,1,1,1,1,1
21,NNA,4,4,4,4,4,4
22,NNB,9,9,9,9,9,9


In [5]:
s=grp['start'].sum()
print(s)
grp['percent']=np.round_(grp['start']*100/s,decimals=2)
#grp

25994


In [6]:
n_c={
'A':'red',
'B':'blue',
'N':'grey' 
}
nodes=[]
maps=OrderedDict()
m=0
for f in ds:
    n=f.replace("OU-HiC-","")
    vals=flips[n].sort_values().unique()
    res = dict(zip(vals, np.arange(m,m+len(vals))))
    maps[n]=res
    cols = np.array([n_c[k[-1]] for k in vals])
    #print(cols)
    a = np.array(list(zip(np.arange(m,m+len(vals)),[x[-1] for x in vals],cols ))).tolist()
    for x in a:
        nodes.append(x)
    #print(a)
    m=m+len(vals)
#nodes

In [7]:
colors =  {
'A':'salmon',
'B':'lightblue',
'N':'lightgrey' 
}

In [8]:
d1 = ds[0].replace("OU-HiC-","")
d2 = ds[1].replace("OU-HiC-","")
Node1 = flips.groupby([d1, d2], as_index=False).count()
Node1 = Node1.filter([d1, d2, 'change']) 
Node1.insert(3, column='Link Color', value='') 
for index, row in Node1.iterrows():
    Node1.at[index,'Link Color'] = colors[row[d1][-1]]
Node1 = Node1.replace({d1: maps[d1],d2: maps[d2]})
Node1['percent']=Node1['change']*100/EVlen
print(Node1['change'].sum())
print(Node1['percent'].sum())
#Node1

25994
90.38247566063978


/tmp/ipykernel_10285/1635080579.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Node1 = Node1.replace({d1: maps[d1],d2: maps[d2]})


In [9]:
d1 = ds[1].replace("OU-HiC-","")
d2 = ds[2].replace("OU-HiC-","")
Node2 = flips.groupby([d1, d2], as_index=False).count()
Node2 = Node2.filter([d1, d2, 'change']) 
Node2.insert(3, column='Link Color', value='') 
for index, row in Node2.iterrows():
    Node2.at[index,'Link Color'] = colors[row[d1][-1]]
Node2 = Node2.replace({d1: maps[d1],d2: maps[d2] })
Node2['percent']=Node2['change']*100/EVlen
print(Node2['change'].sum())
print(Node2['percent'].sum())
#Node2

25994
90.38247566063977


/tmp/ipykernel_10285/3135962138.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Node2 = Node2.replace({d1: maps[d1],d2: maps[d2] })


In [10]:
Node1 = Node1.values
Node2 = Node2.values
nodes=nodes

In [11]:
columns = ['Source','Target','Value', 'Link Color','percent']

# Convert array to dataframe
link1 = pd.DataFrame(Node1, columns = columns )
link2 = pd.DataFrame(Node2, columns = columns)


# Make one dataframe from the 3 dataframes - Union all 3 df
links_df = pd.DataFrame()
links_df = pd.concat([links_df, link1], ignore_index=True)
links_df = pd.concat([links_df,link2], ignore_index=True)

links = links_df
#links

In [12]:
# Retrieve headers and build dataframes

nodes_headers = ['ID', 'Label', 'Color']
links_headers = ['Source','Target','Value','Link Color','percent']
df_nodes = pd.DataFrame(nodes, columns = nodes_headers)
df_links = pd.DataFrame(links, columns = links_headers)

In [13]:
df_nodes.head()

,ID,Label,Color
0,0,A,red
1,1,B,blue
2,2,N,grey
3,3,A,red
4,4,B,blue


In [14]:
df_links.head()

,Source,Target,Value,Link Color,percent
0,0,3,8602,salmon,29.909597
1,0,4,3592,salmon,12.489569
2,0,5,2,salmon,0.006954
3,1,6,4250,lightblue,14.777469
4,1,7,9430,lightblue,32.788595


In [15]:
import pandas as pd
import numpy as np
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
from random import seed
from random import randint

In [16]:
import plotly.graph_objects as go
import urllib, json


fig = go.Figure(data=[go.Sankey(
       valueformat = '0f',
        valuesuffix = "%",
    # Define nodes
    node = dict(
      pad = 10,
    # thickness = 30,
      line = dict(
        color = "white",
        width = 0
      ),
      label =  df_nodes['Label'].dropna(axis=0, how='any'),
      color = df_nodes['Color']
    ),
    # Add links
    link = dict(
      source = df_links['Source'].dropna(axis=0, how='any'),
      target = df_links['Target'].dropna(axis=0, how='any'),
      value = df_links['Value'].dropna(axis=0, how='any'),
      color = df_links['Link Color'].dropna(axis=0, how='any'),
        label = df_links['percent'].dropna(axis=0, how='any'),
        hovertemplate='%{label:.2f} %'+'<extra></extra>',
))])

#fig.update_layout(title_text="Flips(100kb arms): Fibro-IPSC-W0-W3-W6",font_size=10)
fig.update_layout(title_text="Flips(100kb arms): Fibro-IPSC-MN",font_size=10)
fig.write_html("GitHub/Flips-allEVs-3stages-N6-100kb-arms.html")